# Game IP Collaboration Fit Scoring Model

A public, reusable notebook for evaluating whether a game should collaborate with a candidate IP.

This notebook is designed as a **generic scoring framework**. It uses fictional sample data and placeholder IPs, so it can be shared publicly on GitHub without exposing any internal project information.

## What this model does

The model evaluates candidate collaboration IPs using:

- Core user overlap
- Channel / community activity overlap
- High-engagement / influencer-style user overlap
- Purchase intent signals
- Collaboration expectation signals
- Sentiment and fit-risk signals
- UGC / creator-discussion signals
- Manual business priors
- Optional human reviewer scores

## Two final scores

1. `final_score_auto`
   - Automated feature-based score.
   - Suitable for first-pass screening.

2. `final_score_human_weighted`
   - A blend of automated score and weighted reviewer scores.
   - Suitable for final review or stakeholder discussion.

## Plain-language explanation

Instead of asking “Is this IP popular?”, this model asks:

> Are the game’s core users also active fans of the candidate IP?

That distinction matters because a collaboration is usually stronger when the current community naturally overlaps with the IP’s audience.

## 1. Install and import dependencies

This cell installs and imports the Python packages used for tables, scoring, plotting, API templates, and Excel export.

Plain-language note: this is just setting up the toolbox before the analysis starts.

In [ ]:
!pip install pandas numpy openpyxl matplotlib tqdm requests

import pandas as pd
import numpy as np
import re
import math
import requests
import time
from datetime import datetime, timezone
import matplotlib.pyplot as plt
from tqdm import tqdm

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 260)


## 2. Project and candidate IP configuration

This section defines the game being evaluated and the candidate collaboration IPs.

Each object has:

- `interest`: the standard name used in tables and reports
- `keywords`: words or phrases used to detect related discussion
- `channels`: community spaces that represent direct activity for that interest

Plain-language note: this section is the model’s dictionary. It tells the notebook what counts as discussion of the base game and what counts as discussion of each candidate IP.

In [ ]:
base_game_config = {
    "interest": "Base Game",
    "keywords": ["Base Game", "BG", "Astra City", "Skyline Quest"],
    "channels": ["base-game", "bg", "astra-city", "skyline-quest"],
    "x_accounts": [],
    "reddit_subreddits": []
}

candidate_ips = [
    {
        "interest": "Mecha Legends",
        "keywords": ["Mecha Legends", "MLG", "Pilot Zero", "Silver Frame", "mecha anime"],
        "channels": ["mecha-legends", "mlg", "mecha-anime"],
        "x_accounts": [],
        "reddit_subreddits": ["mechaanime"]
    },
    {
        "interest": "Neon Detectives",
        "keywords": ["Neon Detectives", "ND", "Detective Nova", "Neon Case", "cyber mystery"],
        "channels": ["neon-detectives", "nd", "cyber-mystery"],
        "x_accounts": [],
        "reddit_subreddits": ["mysterygames"]
    },
    {
        "interest": "Dragon Academy",
        "keywords": ["Dragon Academy", "DA", "Dragon Mage", "Arcane Dragon", "fantasy school"],
        "channels": ["dragon-academy", "da", "fantasy-school"],
        "x_accounts": [],
        "reddit_subreddits": ["fantasyanime"]
    }
]

candidate_ip_df = pd.DataFrame(candidate_ips)
candidate_ip_df


## 3. Scoring parameters

This section controls how user activity is weighted.

The model gives weight to:

- Channel activity
- Keyword mentions
- Engagement
- Active days
- Recency
- Content diversity

Plain-language note: a user who repeatedly posts in the base game’s channel, mentions the base game often, gets replies, and stays active over time should count more than someone who mentioned the game once.

In [ ]:
USER_SCORE_WEIGHTS = {
    "channel_activity": 0.30,
    "keyword_mentions": 0.25,
    "engagement": 0.18,
    "active_days": 0.12,
    "recency": 0.08,
    "content_diversity": 0.07
}

ACTIVE_USER_QUANTILE = 0.60
CORE_USER_QUANTILE = 0.80
KOL_USER_QUANTILE = 0.90

MIN_ACTIVE_SCORE = 10
MIN_CORE_SCORE = 20
MIN_KOL_SCORE = 30

AUTO_WEIGHT_IN_HUMAN_WEIGHTED_SCORE = 0.70

SCORE_WEIGHTS = {
    "Audience Overlap": 25,
    "Content Fit": 20,
    "Commercial Potential": 20,
    "Reputation Safety": 15,
    "Community Spread": 10,
    "Execution Feasibility": 10
}

SCORE_WEIGHTS


## 4. Standard data structure

The notebook standardizes all data into two tables:

### `content_df`

Stores raw posts, comments, or messages.

### `user_interest_df`

Stores the interest signals extracted from those posts, comments, or messages.

Plain-language note: `content_df` is what users said. `user_interest_df` is what the model understood from what they said.

In [ ]:
STANDARD_CONTENT_COLUMNS = [
    "content_id", "user_id", "platform", "channel", "text", "likes", "replies", "views", "created_at", "source"
]

STANDARD_USER_INTEREST_COLUMNS = [
    "user_id", "platform", "interest", "signal_type", "strength", "channel", "observed_at", "source"
]

def now_iso():
    return datetime.now(timezone.utc).isoformat()

def safe_num(x, default=0):
    try:
        if pd.isna(x):
            return default
        return float(x)
    except Exception:
        return default

def normalize_user_id(platform, raw_id):
    if pd.isna(raw_id) or str(raw_id).strip() == "":
        return None
    return f"{platform}:{str(raw_id).strip().lower()}"

def normalize_channel_name(x):
    if pd.isna(x):
        return ""
    s = str(x).strip().lower()
    s = s.replace("#", "")
    return s

def empty_content_df():
    return pd.DataFrame(columns=STANDARD_CONTENT_COLUMNS)

def empty_user_interest_df():
    return pd.DataFrame(columns=STANDARD_USER_INTEREST_COLUMNS)

def make_content_records(records):
    df = pd.DataFrame(records)
    for col in STANDARD_CONTENT_COLUMNS:
        if col not in df.columns:
            df[col] = None
    df = df[STANDARD_CONTENT_COLUMNS]
    df["likes"] = pd.to_numeric(df["likes"], errors="coerce").fillna(0)
    df["replies"] = pd.to_numeric(df["replies"], errors="coerce").fillna(0)
    df["views"] = pd.to_numeric(df["views"], errors="coerce").fillna(0)
    df["channel"] = df["channel"].apply(normalize_channel_name)
    return df.drop_duplicates()

def make_user_interest_records(records):
    df = pd.DataFrame(records)
    for col in STANDARD_USER_INTEREST_COLUMNS:
        if col not in df.columns:
            df[col] = None
    df = df[STANDARD_USER_INTEREST_COLUMNS]
    df = df.dropna(subset=["user_id", "interest"])
    df["strength"] = pd.to_numeric(df["strength"], errors="coerce").fillna(1)
    df["channel"] = df["channel"].apply(normalize_channel_name)
    return df.drop_duplicates()

def merge_content_tables(tables):
    valid = [t for t in tables if t is not None and not t.empty]
    if not valid:
        return empty_content_df()
    return pd.concat(valid, ignore_index=True).drop_duplicates()

def merge_user_interest_tables(tables):
    valid = [t for t in tables if t is not None and not t.empty]
    if not valid:
        return empty_user_interest_df()
    return pd.concat(valid, ignore_index=True).drop_duplicates()


# Part A: Data ingestion

## 5. CSV connector

This is the recommended first step for public or early-stage usage.

You can export Discord messages, forum posts, Reddit data, YouTube comments, or other community data as CSV, then feed it into this connector.

Plain-language note: this block turns a spreadsheet of community posts into the two standard tables the model needs.

In [ ]:
def find_first_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

def keyword_pattern(keywords):
    kws = [re.escape(str(k).lower()) for k in keywords if str(k).strip()]
    if not kws:
        return None
    return re.compile("|".join(kws), re.IGNORECASE)

def text_contains_any(text, keywords):
    pat = keyword_pattern(keywords)
    if pat is None:
        return False
    return bool(pat.search(str(text)))

def keyword_hit_count(text, keywords):
    t = str(text).lower()
    return sum(len(re.findall(re.escape(str(k).lower()), t)) for k in keywords if str(k).strip())

def channel_matches(channel, configured_channels):
    ch = normalize_channel_name(channel)
    cfg = [normalize_channel_name(c) for c in configured_channels]
    return ch in cfg

def csv_connector(
    csv_path,
    platform,
    interest_configs,
    text_cols=("text", "content", "message", "body", "comment", "title", "Content"),
    author_cols=("user_id", "author", "user", "username", "name", "AuthorID", "Author"),
    channel_cols=("channel", "channel_name", "Channel", "ChannelName", "room", "forum", "subreddit"),
    like_cols=("likes", "score", "upvotes", "like_count", "Reactions", "reactions"),
    reply_cols=("replies", "comments", "num_comments", "reply_count", "Replies"),
    view_cols=("views", "view_count", "plays"),
    created_cols=("created_at", "timestamp", "time", "date", "Date")
):
    raw = pd.read_csv(csv_path)

    text_col = find_first_col(raw, text_cols)
    author_col = find_first_col(raw, author_cols)
    channel_col = find_first_col(raw, channel_cols)
    like_col = find_first_col(raw, like_cols)
    reply_col = find_first_col(raw, reply_cols)
    view_col = find_first_col(raw, view_cols)
    created_col = find_first_col(raw, created_cols)

    if text_col is None or author_col is None:
        raise ValueError("CSV must contain at least a text column and an author/user column.")

    content_records = []
    interest_records = []

    for idx, row in raw.iterrows():
        user_id = normalize_user_id(platform, row.get(author_col))
        if user_id is None:
            continue

        text = str(row.get(text_col, ""))
        channel = normalize_channel_name(row.get(channel_col, "")) if channel_col else ""
        likes = safe_num(row.get(like_col, 0)) if like_col else 0
        replies = safe_num(row.get(reply_col, 0)) if reply_col else 0
        views = safe_num(row.get(view_col, 0)) if view_col else 0
        created_at = row.get(created_col, None) if created_col else None
        engagement = likes + replies * 2 + views * 0.01
        base_strength = 1 + math.log1p(max(0, engagement))

        content_records.append({
            "content_id": f"{platform}:csv:{idx}",
            "user_id": user_id,
            "platform": platform,
            "channel": channel,
            "text": text,
            "likes": likes,
            "replies": replies,
            "views": views,
            "created_at": created_at,
            "source": csv_path
        })

        for cfg in interest_configs:
            hits = keyword_hit_count(text, cfg.get("keywords", []))
            if hits > 0:
                interest_records.append({
                    "user_id": user_id,
                    "platform": platform,
                    "interest": cfg["interest"],
                    "signal_type": "keyword_mention",
                    "strength": base_strength * hits,
                    "channel": channel,
                    "observed_at": created_at if created_at is not None else now_iso(),
                    "source": csv_path
                })

            if channel and channel_matches(channel, cfg.get("channels", [])):
                interest_records.append({
                    "user_id": user_id,
                    "platform": platform,
                    "interest": cfg["interest"],
                    "signal_type": "channel_activity",
                    "strength": base_strength,
                    "channel": channel,
                    "observed_at": created_at if created_at is not None else now_iso(),
                    "source": csv_path
                })

    return make_user_interest_records(interest_records), make_content_records(content_records)

# Example usage:
# from google.colab import files
# uploaded = files.upload()
# csv_path = list(uploaded.keys())[0]
# configs = [base_game_config] + candidate_ips
# ui_csv, content_csv = csv_connector(csv_path, platform="discord", interest_configs=configs)


## 6. API connector templates

This section includes a simple Reddit-style API template. For most teams, CSV ingestion is easier to validate first.

Plain-language note: APIs are useful later. The important thing is that every connector should eventually return `content_df` and `user_interest_df` in the same format.

In [ ]:
HEADERS = {"User-Agent": "game-ip-collaboration-fit-model/1.0"}

def safe_get_json(url, params=None, headers=HEADERS, timeout=15, sleep=1.0):
    try:
        r = requests.get(url, params=params, headers=headers, timeout=timeout)
        time.sleep(sleep)
        if r.status_code == 200:
            return r.json()
        return None
    except Exception:
        return None

def reddit_search_posts(query, subreddit=None, limit=50, time_filter="month"):
    if subreddit:
        url = f"https://www.reddit.com/r/{subreddit}/search.json"
        params = {"q": query, "restrict_sr": "on", "sort": "new", "t": time_filter, "limit": limit}
    else:
        url = "https://www.reddit.com/search.json"
        params = {"q": query, "sort": "new", "t": time_filter, "limit": limit}
    data = safe_get_json(url, params=params)
    if not data:
        return []
    posts = []
    for child in data.get("data", {}).get("children", []):
        d = child.get("data", {})
        posts.append({
            "id": d.get("id"),
            "author": d.get("author"),
            "title": d.get("title", ""),
            "selftext": d.get("selftext", ""),
            "score": d.get("score", 0),
            "num_comments": d.get("num_comments", 0),
            "created_utc": d.get("created_utc"),
            "subreddit": d.get("subreddit")
        })
    return posts

def reddit_connector_for_interest(interest_config, subreddits=None, limit_per_query=30):
    records_interest = []
    records_content = []
    subreddits = subreddits or interest_config.get("reddit_subreddits", []) or [None]
    for kw in interest_config.get("keywords", []):
        for sr in subreddits:
            posts = reddit_search_posts(kw, subreddit=sr, limit=limit_per_query, time_filter="month")
            for p in posts:
                if not p.get("author") or p.get("author") == "[deleted]":
                    continue
                user_id = normalize_user_id("reddit", p["author"])
                channel = normalize_channel_name(p.get("subreddit", ""))
                text = f"{p.get('title','')} {p.get('selftext','')}"
                likes = safe_num(p.get("score", 0))
                replies = safe_num(p.get("num_comments", 0))
                strength = 1 + math.log1p(max(0, likes + replies * 2))
                records_content.append({
                    "content_id": f"reddit:{p.get('id')}", "user_id": user_id, "platform": "reddit", "channel": channel,
                    "text": text, "likes": likes, "replies": replies, "views": 0, "created_at": p.get("created_utc"),
                    "source": f"reddit_search:{kw}:{sr}"
                })
                records_interest.append({
                    "user_id": user_id, "platform": "reddit", "interest": interest_config["interest"],
                    "signal_type": "keyword_mention", "strength": strength, "channel": channel,
                    "observed_at": p.get("created_utc"), "source": f"reddit_search:{kw}:{sr}"
                })
    return make_user_interest_records(records_interest), make_content_records(records_content)


# Part B: Sample data

## 7. Fictional demo data

The following sample data is fictional and safe for public sharing.

Plain-language note: this lets the notebook run out of the box. Replace this section with real exported data when using the model for an actual analysis.

In [ ]:
demo_content = make_content_records([
    {"content_id": "c1", "user_id": "discord:u1", "platform": "discord", "channel": "base-game", "text": "Base Game city exploration feels great, Astra City looks promising", "likes": 8, "replies": 3, "views": 0, "created_at": "2026-06-01", "source": "demo"},
    {"content_id": "c2", "user_id": "discord:u1", "platform": "discord", "channel": "base-game", "text": "I want Base Game x Mecha Legends collab, Pilot Zero skin would be perfect, I would buy", "likes": 20, "replies": 6, "views": 0, "created_at": "2026-06-02", "source": "demo"},
    {"content_id": "c3", "user_id": "discord:u1", "platform": "discord", "channel": "mecha-legends", "text": "Mecha Legends fits Base Game vibes, Silver Frame is iconic", "likes": 12, "replies": 4, "views": 0, "created_at": "2026-06-03", "source": "demo"},

    {"content_id": "c4", "user_id": "discord:u2", "platform": "discord", "channel": "base-game", "text": "Base Game housing system is interesting", "likes": 6, "replies": 1, "views": 0, "created_at": "2026-06-01", "source": "demo"},
    {"content_id": "c5", "user_id": "discord:u2", "platform": "discord", "channel": "base-game", "text": "Base Game open world and story look promising", "likes": 7, "replies": 2, "views": 0, "created_at": "2026-06-02", "source": "demo"},

    {"content_id": "c6", "user_id": "forum:r1", "platform": "forum", "channel": "general-games", "text": "Base Game looks nice but Neon Detectives collab feels weird and forced", "likes": 15, "replies": 8, "views": 0, "created_at": "2026-06-01", "source": "demo"},
    {"content_id": "c7", "user_id": "forum:r1", "platform": "forum", "channel": "neon-detectives", "text": "Neon Detectives cases are cool, Detective Nova has strong fanart potential", "likes": 10, "replies": 3, "views": 0, "created_at": "2026-06-02", "source": "demo"},

    {"content_id": "c8", "user_id": "youtube:y1", "platform": "youtube", "channel": "comments", "text": "Base Game reminds me of fantasy school anime", "likes": 4, "replies": 1, "views": 0, "created_at": "2026-06-01", "source": "demo"},
    {"content_id": "c9", "user_id": "youtube:y1", "platform": "youtube", "channel": "comments", "text": "Dragon Academy collab could be hype, Dragon Mage outfit would be cool", "likes": 12, "replies": 2, "views": 0, "created_at": "2026-06-03", "source": "demo"},

    {"content_id": "c10", "user_id": "discord:u3", "platform": "discord", "channel": "mecha-legends", "text": "Pilot Zero and Silver Frame are iconic", "likes": 5, "replies": 1, "views": 0, "created_at": "2026-06-01", "source": "demo"},
    {"content_id": "c11", "user_id": "discord:u4", "platform": "discord", "channel": "dragon-academy", "text": "Dragon Mage and Arcane Dragon are classic", "likes": 5, "replies": 1, "views": 0, "created_at": "2026-06-01", "source": "demo"},
])

def infer_user_interest_from_content(content_df, interest_configs):
    records = []
    for _, row in content_df.iterrows():
        engagement = safe_num(row.get("likes", 0)) + safe_num(row.get("replies", 0)) * 2 + safe_num(row.get("views", 0)) * 0.01
        strength = 1 + math.log1p(max(0, engagement))
        text = row.get("text", "")
        channel = row.get("channel", "")
        for cfg in interest_configs:
            hits = keyword_hit_count(text, cfg.get("keywords", []))
            if hits > 0:
                records.append({
                    "user_id": row["user_id"], "platform": row["platform"], "interest": cfg["interest"],
                    "signal_type": "keyword_mention", "strength": strength * hits, "channel": channel,
                    "observed_at": row.get("created_at"), "source": row.get("source")
                })
            if channel and channel_matches(channel, cfg.get("channels", [])):
                records.append({
                    "user_id": row["user_id"], "platform": row["platform"], "interest": cfg["interest"],
                    "signal_type": "channel_activity", "strength": strength, "channel": channel,
                    "observed_at": row.get("created_at"), "source": row.get("source")
                })
    return make_user_interest_records(records)

content_df = demo_content.copy()
user_interest_df = infer_user_interest_from_content(content_df, [base_game_config] + candidate_ips)

print("Extracted user-interest signals:")
display(user_interest_df)
print("Raw content table:")
display(content_df)


# Part C: User strength scoring

## 8. Calculate each user’s interest strength

This section turns raw activity into a user-level interest score.

Plain-language note: this answers “How strongly does this user appear to care about this game or IP?”

In [ ]:
def parse_date_safe(x):
    try:
        return pd.to_datetime(x, errors="coerce", utc=True)
    except Exception:
        return pd.NaT

def scale_0_100(s):
    s = pd.Series(s).astype(float).replace([np.inf, -np.inf], np.nan).fillna(0)
    if s.max() == s.min():
        return pd.Series([50 if s.max() > 0 else 0] * len(s), index=s.index)
    return (s - s.min()) / (s.max() - s.min()) * 100

def log_scale_0_100(s):
    return scale_0_100(np.log1p(pd.Series(s).astype(float).fillna(0).clip(lower=0)))

def build_user_interest_strength(content_df, user_interest_df):
    ci = user_interest_df.copy()
    if ci.empty:
        return pd.DataFrame()

    ci["observed_dt"] = ci["observed_at"].apply(parse_date_safe)
    ci["date"] = ci["observed_dt"].dt.date

    rows = []
    for interest in ci["interest"].dropna().unique():
        sub = ci[ci["interest"] == interest].copy()
        for user_id, g in sub.groupby("user_id"):
            platform = g["platform"].mode().iloc[0] if not g["platform"].mode().empty else None
            keyword_mentions = g.loc[g["signal_type"] == "keyword_mention", "strength"].sum()
            channel_activity = g.loc[g["signal_type"] == "channel_activity", "strength"].sum()
            keyword_event_count = (g["signal_type"] == "keyword_mention").sum()
            channel_event_count = (g["signal_type"] == "channel_activity").sum()
            total_signals = len(g)
            raw_strength = g["strength"].sum()
            active_days = g["date"].nunique() if "date" in g else 0
            channel_diversity = g["channel"].nunique()
            last_seen = g["observed_dt"].max()

            if pd.isna(last_seen):
                recency_raw = 0.5
            else:
                now = pd.Timestamp.now(tz="UTC")
                days_ago = max(0, (now - last_seen).days)
                recency_raw = math.exp(-days_ago / 30)

            rows.append({
                "user_id": user_id,
                "platform": platform,
                "interest": interest,
                "keyword_mentions_strength": keyword_mentions,
                "channel_activity_strength": channel_activity,
                "keyword_event_count": keyword_event_count,
                "channel_event_count": channel_event_count,
                "total_signals": total_signals,
                "raw_strength": raw_strength,
                "active_days": active_days,
                "channel_diversity": channel_diversity,
                "recency_raw": recency_raw,
                "last_seen": last_seen
            })

    df = pd.DataFrame(rows)
    if df.empty:
        return df

    scored = []
    for interest, g in df.groupby("interest"):
        g = g.copy()
        g["channel_activity_score"] = log_scale_0_100(g["channel_activity_strength"])
        g["keyword_mentions_score"] = log_scale_0_100(g["keyword_mentions_strength"])
        g["engagement_score"] = log_scale_0_100(g["raw_strength"])
        g["active_days_score"] = log_scale_0_100(g["active_days"])
        g["recency_score"] = scale_0_100(g["recency_raw"])
        g["content_diversity_score"] = log_scale_0_100(g["channel_diversity"])

        g["interest_strength_score"] = (
            g["channel_activity_score"] * USER_SCORE_WEIGHTS["channel_activity"] +
            g["keyword_mentions_score"] * USER_SCORE_WEIGHTS["keyword_mentions"] +
            g["engagement_score"] * USER_SCORE_WEIGHTS["engagement"] +
            g["active_days_score"] * USER_SCORE_WEIGHTS["active_days"] +
            g["recency_score"] * USER_SCORE_WEIGHTS["recency"] +
            g["content_diversity_score"] * USER_SCORE_WEIGHTS["content_diversity"]
        ).round(2)

        scored.append(g)

    return pd.concat(scored, ignore_index=True).sort_values(["interest", "interest_strength_score"], ascending=[True, False]).reset_index(drop=True)

user_strength_df = build_user_interest_strength(content_df, user_interest_df)
user_strength_df


## 9. Segment users into tiers

The model classifies users into regular, active, core, and high-engagement users.

Plain-language note: not every user should count equally. Core users and high-engagement users usually matter more for collaboration fit.

In [ ]:
def assign_user_tiers(user_strength_df):
    if user_strength_df.empty:
        return user_strength_df
    out = []
    for interest, g in user_strength_df.groupby("interest"):
        g = g.copy()
        active_cut = max(MIN_ACTIVE_SCORE, g["interest_strength_score"].quantile(ACTIVE_USER_QUANTILE))
        core_cut = max(MIN_CORE_SCORE, g["interest_strength_score"].quantile(CORE_USER_QUANTILE))
        kol_cut = max(MIN_KOL_SCORE, g["interest_strength_score"].quantile(KOL_USER_QUANTILE))

        g["is_active_user"] = g["interest_strength_score"] >= active_cut
        g["is_core_user"] = g["interest_strength_score"] >= core_cut
        g["is_high_engagement_user"] = g["interest_strength_score"] >= kol_cut
        g["active_cutoff"] = round(active_cut, 2)
        g["core_cutoff"] = round(core_cut, 2)
        g["high_engagement_cutoff"] = round(kol_cut, 2)

        def tier(row):
            if row["is_high_engagement_user"]:
                return "High-engagement user"
            if row["is_core_user"]:
                return "Core user"
            if row["is_active_user"]:
                return "Active user"
            return "Regular user"

        g["user_tier"] = g.apply(tier, axis=1)
        out.append(g)
    return pd.concat(out, ignore_index=True)

user_tier_df = assign_user_tiers(user_strength_df)
user_tier_df


# Part D: Audience overlap algorithm

## 10. Calculate overlap between base-game users and candidate-IP users

This is the core model logic.

The model measures:

- Basic overlap
- Active-user overlap
- Core-user overlap
- Weighted core-user overlap
- High-engagement-user overlap
- Channel-activity overlap
- Lift versus base rate

Plain-language note: this section answers whether the base game’s strongest users are also strong users of the candidate IP.

In [ ]:
def interest_users(user_tier_df, interest, filter_col=None):
    sub = user_tier_df[user_tier_df["interest"] == interest]
    if filter_col:
        sub = sub[sub[filter_col] == True]
    return set(sub["user_id"].dropna())

def channel_active_users(user_interest_df, interest):
    sub = user_interest_df[(user_interest_df["interest"] == interest) & (user_interest_df["signal_type"] == "channel_activity")]
    return set(sub["user_id"].dropna())

def weighted_overlap_rate(user_tier_df, base_interest, target_interest, base_filter_col="is_core_user", target_filter_col="is_core_user"):
    base = user_tier_df[(user_tier_df["interest"] == base_interest) & (user_tier_df[base_filter_col] == True)].copy()
    target_users = set(user_tier_df[(user_tier_df["interest"] == target_interest) & (user_tier_df[target_filter_col] == True)]["user_id"])
    if base.empty or base["interest_strength_score"].sum() <= 0:
        return 0
    overlap = base[base["user_id"].isin(target_users)]
    return overlap["interest_strength_score"].sum() / base["interest_strength_score"].sum()

def rate(a, b):
    return len(a & b) / len(a) if len(a) > 0 else 0

def jaccard(a, b):
    return len(a & b) / len(a | b) if len(a | b) > 0 else 0

def compute_core_overlap_features(user_tier_df, user_interest_df, base_interest, target_interest):
    all_users_global = set(user_tier_df["user_id"].dropna())

    base_all = interest_users(user_tier_df, base_interest)
    target_all = interest_users(user_tier_df, target_interest)

    base_active = interest_users(user_tier_df, base_interest, "is_active_user")
    target_active = interest_users(user_tier_df, target_interest, "is_active_user")

    base_core = interest_users(user_tier_df, base_interest, "is_core_user")
    target_core = interest_users(user_tier_df, target_interest, "is_core_user")

    base_high = interest_users(user_tier_df, base_interest, "is_high_engagement_user")
    target_high = interest_users(user_tier_df, target_interest, "is_high_engagement_user")

    base_channel = channel_active_users(user_interest_df, base_interest)
    target_channel = channel_active_users(user_interest_df, target_interest)

    target_core_base_rate = len(target_core) / len(all_users_global) if all_users_global else 0
    core_overlap_rate = rate(base_core, target_core)
    lift_core_vs_base = core_overlap_rate / target_core_base_rate if target_core_base_rate > 0 else 0

    return {
        "target_interest": target_interest,
        "base_user_count": len(base_all),
        "ip_user_count": len(target_all),
        "basic_overlap_count": len(base_all & target_all),
        "basic_overlap_rate": rate(base_all, target_all),
        "basic_jaccard": jaccard(base_all, target_all),

        "base_active_count": len(base_active),
        "ip_active_count": len(target_active),
        "active_overlap_count": len(base_active & target_active),
        "active_overlap_rate": rate(base_active, target_active),

        "base_core_count": len(base_core),
        "ip_core_count": len(target_core),
        "core_overlap_count": len(base_core & target_core),
        "core_overlap_rate": core_overlap_rate,
        "weighted_core_overlap_rate": weighted_overlap_rate(user_tier_df, base_interest, target_interest, "is_core_user", "is_core_user"),

        "base_high_engagement_count": len(base_high),
        "ip_high_engagement_count": len(target_high),
        "high_engagement_overlap_count": len(base_high & target_high),
        "high_engagement_overlap_rate": rate(base_high, target_high),

        "base_channel_active_count": len(base_channel),
        "ip_channel_active_count": len(target_channel),
        "channel_overlap_count": len(base_channel & target_channel),
        "channel_overlap_rate": rate(base_channel, target_channel),

        "target_core_base_rate": target_core_base_rate,
        "lift_core_vs_base": lift_core_vs_base
    }

core_overlap_df = pd.DataFrame([
    compute_core_overlap_features(user_tier_df, user_interest_df, "Base Game", ip["interest"])
    for ip in candidate_ips
])
core_overlap_df


## 11. Show overlap evidence

This block shows which users are core users of both the base game and a candidate IP.

Plain-language note: this is a sanity-check block. It helps you inspect whether the model is identifying real overlap or just keyword noise.

In [ ]:
def show_core_overlap_evidence(user_tier_df, content_df, base_interest="Base Game", target_interests=None, max_users_per_ip=10, max_texts_per_user=5):
    if target_interests is None:
        target_interests = sorted([x for x in user_tier_df["interest"].dropna().unique() if x != base_interest])

    evidence_rows = []
    base_core = set(user_tier_df[(user_tier_df["interest"] == base_interest) & (user_tier_df["is_core_user"] == True)]["user_id"])

    for target in target_interests:
        target_core = set(user_tier_df[(user_tier_df["interest"] == target) & (user_tier_df["is_core_user"] == True)]["user_id"])
        overlap_users = sorted(list(base_core & target_core))

        print("=" * 110)
        print(f"Candidate IP: {target}")
        print(f"Base-game core users: {len(base_core)}")
        print(f"{target} core users: {len(target_core)}")
        print(f"Users who are core users of both: {len(overlap_users)}")

        if not overlap_users:
            print("No overlapping core users found.")
            continue

        for user_id in overlap_users[:max_users_per_ip]:
            print("-" * 110)
            print(f"{user_id} is a core user of both {base_interest} and {target}")

            tier_rows = user_tier_df[(user_tier_df["user_id"] == user_id) & (user_tier_df["interest"].isin([base_interest, target]))]
            display(tier_rows[["user_id", "platform", "interest", "interest_strength_score", "user_tier", "keyword_event_count", "channel_event_count", "active_days", "raw_strength"]])

            texts = content_df[content_df["user_id"] == user_id].copy()
            display_cols = ["platform", "channel", "text", "likes", "replies", "created_at", "source"]
            display_cols = [c for c in display_cols if c in texts.columns]
            display(texts[display_cols].head(max_texts_per_user))

            for _, r in texts.head(max_texts_per_user).iterrows():
                evidence_rows.append({
                    "target_interest": target,
                    "user_id": user_id,
                    "platform": r.get("platform"),
                    "channel": r.get("channel"),
                    "text": r.get("text"),
                    "likes": r.get("likes"),
                    "replies": r.get("replies"),
                    "created_at": r.get("created_at"),
                    "note": f"{user_id} is a core user of both {base_interest} and {target}"
                })

    return pd.DataFrame(evidence_rows)

core_overlap_evidence_df = show_core_overlap_evidence(
    user_tier_df=user_tier_df,
    content_df=content_df,
    base_interest="Base Game",
    target_interests=[ip["interest"] for ip in candidate_ips],
    max_users_per_ip=10,
    max_texts_per_user=5
)

print("Core-overlap evidence summary:")
display(core_overlap_evidence_df)


# Part E: Text features

## 12. Sentiment, purchase intent, collaboration expectation, and risk signals

This section detects lightweight keyword-based signals in user text.

Plain-language note: overlap is not enough. You also want to know whether users sound excited, willing to spend, skeptical, or worried that the collaboration feels forced.

In [ ]:
positive_words = [
    "good", "great", "amazing", "love", "like", "hype", "excited", "perfect", "cool", "best", "want", "fits", "iconic"
]
negative_words = [
    "bad", "hate", "boring", "cringe", "trash", "cash grab", "controversy", "scam", "boycott", "weird"
]
purchase_intent_words = [
    "pull", "gacha", "buy", "spend", "banner", "skin", "pass", "bundle", "limited", "rerun", "top up"
]
collab_expect_words = [
    "collab", "crossover", "event", "partnership", "want", "hope", "would be perfect", "dream collab"
]
fit_risk_words = [
    "weird", "forced", "out of place", "doesn't fit", "cash grab"
]
ugc_words = [
    "fanart", "cosplay", "meme", "edit", "clip", "amv", "wallpaper", "fan art"
]
creator_words = [
    "streamer", "youtuber", "creator", "influencer", "guide", "review", "reaction", "livestream"
]

def count_word_hits(text, words):
    t = str(text).lower()
    return sum(1 for w in words if w.lower() in t)

def compute_text_features_for_ip(content_df, interest_config):
    empty = {
        "target_interest": interest_config["interest"],
        "mention_content_count": 0,
        "positive_hits": 0,
        "negative_hits": 0,
        "purchase_intent_hits": 0,
        "collab_expect_hits": 0,
        "fit_risk_hits": 0,
        "ugc_hits": 0,
        "creator_discussion_hits": 0,
        "positive_rate": 0,
        "negative_rate": 0,
        "purchase_intent_rate": 0,
        "collab_expect_rate": 0,
        "fit_risk_rate": 0,
        "ugc_rate": 0,
        "creator_discussion_rate": 0,
        "ip_total_engagement": 0
    }
    if content_df.empty:
        return empty
    pat = keyword_pattern(interest_config.get("keywords", []))
    if pat is None:
        return empty
    matched = content_df[content_df["text"].astype(str).str.contains(pat, na=False)].copy()
    if matched.empty:
        return empty
    matched["engagement"] = matched["likes"].fillna(0) + matched["replies"].fillna(0) * 2 + matched["views"].fillna(0) * 0.01
    pos = matched["text"].apply(lambda x: count_word_hits(x, positive_words)).sum()
    neg = matched["text"].apply(lambda x: count_word_hits(x, negative_words)).sum()
    purchase = matched["text"].apply(lambda x: count_word_hits(x, purchase_intent_words)).sum()
    collab = matched["text"].apply(lambda x: count_word_hits(x, collab_expect_words)).sum()
    fitrisk = matched["text"].apply(lambda x: count_word_hits(x, fit_risk_words)).sum()
    ugc = matched["text"].apply(lambda x: count_word_hits(x, ugc_words)).sum()
    creator = matched["text"].apply(lambda x: count_word_hits(x, creator_words)).sum()
    n = len(matched)
    return {
        "target_interest": interest_config["interest"],
        "mention_content_count": n,
        "positive_hits": float(pos),
        "negative_hits": float(neg),
        "purchase_intent_hits": float(purchase),
        "collab_expect_hits": float(collab),
        "fit_risk_hits": float(fitrisk),
        "ugc_hits": float(ugc),
        "creator_discussion_hits": float(creator),
        "positive_rate": float(pos / max(1, pos + neg)),
        "negative_rate": float(neg / max(1, pos + neg)),
        "purchase_intent_rate": float(purchase / max(1, n)),
        "collab_expect_rate": float(collab / max(1, n)),
        "fit_risk_rate": float(fitrisk / max(1, n)),
        "ugc_rate": float(ugc / max(1, n)),
        "creator_discussion_rate": float(creator / max(1, n)),
        "ip_total_engagement": float(matched["engagement"].sum())
    }

text_feature_df = pd.DataFrame([compute_text_features_for_ip(content_df, ip) for ip in candidate_ips])
text_feature_df


# Part F: Automated scoring

## 13. Convert features into scoring dimensions

The model converts automated features into six dimensions:

- Audience Overlap /25
- Content Fit /20
- Commercial Potential /20
- Reputation Safety /15
- Community Spread /10
- Execution Feasibility /10

Plain-language note: this turns raw metrics into a management-friendly 100-point score.

In [ ]:
feature_df = core_overlap_df.merge(text_feature_df, on="target_interest", how="left").fillna(0)

manual_prior = pd.DataFrame([
    {"target_interest": "Mecha Legends", "content_fit_prior": 88, "execution_prior": 72, "budget_risk": "mid", "legal_risk": "low", "competitor_used": False},
    {"target_interest": "Neon Detectives", "content_fit_prior": 78, "execution_prior": 64, "budget_risk": "mid", "legal_risk": "mid", "competitor_used": False},
    {"target_interest": "Dragon Academy", "content_fit_prior": 82, "execution_prior": 68, "budget_risk": "low", "legal_risk": "low", "competitor_used": True},
])

def risk_penalty(risk):
    risk = str(risk).lower()
    if risk in ["high"]:
        return 35
    if risk in ["mid", "medium"]:
        return 15
    return 0

def build_auto_indices(feature_df):
    df = feature_df.copy()

    df["audience_overlap_index"] = (
        df["core_overlap_rate"].clip(0, 1) * 28 +
        df["weighted_core_overlap_rate"].clip(0, 1) * 22 +
        df["channel_overlap_rate"].clip(0, 1) * 18 +
        df["active_overlap_rate"].clip(0, 1) * 14 +
        df["high_engagement_overlap_rate"].clip(0, 1) * 10 +
        np.minimum(df["lift_core_vs_base"], 5) / 5 * 8
    ).clip(0, 100)

    df["community_acceptance_index"] = (
        df["positive_rate"].clip(0, 1) * 35 +
        df["collab_expect_rate"].clip(0, 1) * 25 +
        (1 - df["fit_risk_rate"].clip(0, 1)) * 25 +
        (1 - df["negative_rate"].clip(0, 1)) * 15
    ).clip(0, 100)

    df["commercial_index"] = (
        scale_0_100(df["purchase_intent_rate"] + df["purchase_intent_hits"] * 0.2) * 0.38 +
        df["audience_overlap_index"] * 0.30 +
        scale_0_100(df["collab_expect_rate"] + df["collab_expect_hits"] * 0.2) * 0.22 +
        log_scale_0_100(df["ip_total_engagement"] + df["mention_content_count"]) * 0.10
    ).clip(0, 100)

    risk_index = (
        scale_0_100(df["negative_rate"] + df["negative_hits"] * 0.2) * 0.45 +
        scale_0_100(df["fit_risk_rate"] + df["fit_risk_hits"] * 0.3) * 0.35 +
        scale_0_100((1 - df["positive_rate"]) + df["negative_rate"]) * 0.20
    ).clip(0, 100)
    df["safety_index_auto"] = (100 - risk_index).clip(0, 100)

    df["spread_index"] = (
        df["high_engagement_overlap_rate"].clip(0, 1) * 25 +
        scale_0_100(df["ugc_rate"] + df["ugc_hits"] * 0.2) * 0.20 +
        scale_0_100(df["creator_discussion_rate"] + df["creator_discussion_hits"] * 0.2) * 0.20 +
        log_scale_0_100(df["ip_total_engagement"] + df["mention_content_count"]) * 0.20 +
        scale_0_100(df["collab_expect_rate"] + df["collab_expect_hits"] * 0.2) * 0.15
    ).clip(0, 100)

    return df

auto_index_df = build_auto_indices(feature_df)
auto_index_df[["target_interest", "audience_overlap_index", "community_acceptance_index", "commercial_index", "safety_index_auto", "spread_index"]]


## 14. Generate the automated score

`final_score_auto` is the feature-based automated score.

Plain-language note: this is the score you can use for first-pass candidate screening before stakeholder review.

In [ ]:
def map_to_auto_score(auto_index_df, manual_prior):
    df = auto_index_df.merge(manual_prior, on="target_interest", how="left")
    df["content_fit_prior"] = df["content_fit_prior"].fillna(60)
    df["execution_prior"] = df["execution_prior"].fillna(60)
    df["budget_risk"] = df["budget_risk"].fillna("mid")
    df["legal_risk"] = df["legal_risk"].fillna("mid")
    df["competitor_used"] = df["competitor_used"].fillna(False)

    df["Audience Overlap"] = (df["audience_overlap_index"] / 100 * 25).round(2)

    content_fit_index = (df["content_fit_prior"] * 0.65 + df["community_acceptance_index"] * 0.35).clip(0, 100)
    df["Content Fit"] = (content_fit_index / 100 * 20).round(2)

    df["Commercial Potential"] = (df["commercial_index"] / 100 * 20).round(2)

    safety_index = (df["safety_index_auto"] - df["legal_risk"].apply(risk_penalty)).clip(0, 100)
    df["Reputation Safety"] = (safety_index / 100 * 15).round(2)

    df["Community Spread"] = (df["spread_index"] / 100 * 10).round(2)

    execution_index = (
        df["execution_prior"]
        - df["budget_risk"].apply(risk_penalty) * 0.55
        - np.where(df["competitor_used"], 8, 0)
    ).clip(0, 100)
    df["Execution Feasibility"] = (execution_index / 100 * 10).round(2)

    score_cols = ["Audience Overlap", "Content Fit", "Commercial Potential", "Reputation Safety", "Community Spread", "Execution Feasibility"]
    df["final_score_auto"] = df[score_cols].sum(axis=1).round(2)
    return df

auto_scored_df = map_to_auto_score(auto_index_df, manual_prior)
auto_scored_df[["target_interest", "final_score_auto", "Audience Overlap", "Content Fit", "Commercial Potential", "Reputation Safety", "Community Spread", "Execution Feasibility"]]


# Part G: Human review scoring

## 15. Reviewer score table

This table is optional. It allows reviewers from market, product, legal, business development, or community teams to contribute structured scores.

Plain-language note: automated scoring is useful, but final decisions often need stakeholder judgment.

In [ ]:
human_scores = pd.DataFrame([
    {"target_interest": "Mecha Legends", "evaluator": "market_reviewer", "role": "market", "Human Audience Fit": 84, "Human Content Fit": 88, "Human Commercial Potential": 86, "Human Spread Potential": 80, "Human Safety": 78, "Human Execution Feasibility": 72, "confidence": 0.9},
    {"target_interest": "Mecha Legends", "evaluator": "product_reviewer", "role": "product", "Human Audience Fit": 82, "Human Content Fit": 90, "Human Commercial Potential": 84, "Human Spread Potential": 78, "Human Safety": 80, "Human Execution Feasibility": 70, "confidence": 0.85},

    {"target_interest": "Neon Detectives", "evaluator": "market_reviewer", "role": "market", "Human Audience Fit": 72, "Human Content Fit": 76, "Human Commercial Potential": 74, "Human Spread Potential": 78, "Human Safety": 65, "Human Execution Feasibility": 66, "confidence": 0.8},
    {"target_interest": "Neon Detectives", "evaluator": "legal_reviewer", "role": "legal", "Human Audience Fit": 70, "Human Content Fit": 74, "Human Commercial Potential": 72, "Human Spread Potential": 76, "Human Safety": 60, "Human Execution Feasibility": 64, "confidence": 0.85},

    {"target_interest": "Dragon Academy", "evaluator": "market_reviewer", "role": "market", "Human Audience Fit": 76, "Human Content Fit": 82, "Human Commercial Potential": 78, "Human Spread Potential": 75, "Human Safety": 80, "Human Execution Feasibility": 74, "confidence": 0.8},
    {"target_interest": "Dragon Academy", "evaluator": "product_reviewer", "role": "product", "Human Audience Fit": 74, "Human Content Fit": 84, "Human Commercial Potential": 76, "Human Spread Potential": 74, "Human Safety": 82, "Human Execution Feasibility": 76, "confidence": 0.85},
])

human_scores


## 16. Compute weighted human score

Reviewer scores are weighted by role and confidence.

Plain-language note: some roles may be more relevant for specific dimensions. Confidence helps account for how certain a reviewer is.

In [ ]:
role_weights = {
    "market": 1.20,
    "product": 1.25,
    "bizdev": 1.00,
    "legal": 1.15,
    "community": 1.10,
    "default": 1.00
}

human_dimension_weights = {
    "Human Audience Fit": 0.25,
    "Human Content Fit": 0.20,
    "Human Commercial Potential": 0.20,
    "Human Safety": 0.15,
    "Human Spread Potential": 0.10,
    "Human Execution Feasibility": 0.10,
}

HUMAN_DIM_COLS = list(human_dimension_weights.keys())

def compute_human_scores(human_scores):
    df = human_scores.copy()
    for c in HUMAN_DIM_COLS:
        df[c] = pd.to_numeric(df[c], errors="coerce").clip(0, 100)
    df["confidence"] = pd.to_numeric(df["confidence"], errors="coerce").fillna(1).clip(0, 1)

    df["human_score_individual"] = 0
    for col, w in human_dimension_weights.items():
        df["human_score_individual"] += df[col] * w

    df["role_weight"] = df["role"].map(role_weights).fillna(role_weights["default"])
    df["final_person_weight"] = df["role_weight"] * df["confidence"]

    agg_rows = []
    for ip, g in df.groupby("target_interest"):
        total_w = g["final_person_weight"].sum()
        weighted_score = (g["human_score_individual"] * g["final_person_weight"]).sum() / total_w if total_w > 0 else g["human_score_individual"].mean()
        row = {
            "target_interest": ip,
            "human_score_weighted": round(weighted_score, 2),
            "human_evaluator_count": len(g),
            "human_total_weight": round(total_w, 2),
            "human_score_std": round(g["human_score_individual"].std(ddof=0), 2),
            "human_score_min": round(g["human_score_individual"].min(), 2),
            "human_score_max": round(g["human_score_individual"].max(), 2),
        }
        for col in HUMAN_DIM_COLS:
            row[f"{col}_weighted"] = round((g[col] * g["final_person_weight"]).sum() / total_w, 2) if total_w > 0 else round(g[col].mean(), 2)
        agg_rows.append(row)
    return pd.DataFrame(agg_rows), df

human_agg_df, human_individual_df = compute_human_scores(human_scores)
human_agg_df


# Part H: Final grading and outputs

## 17. Grade rules and recommendation actions

The model converts final scores into S/A/B/C/D grades and applies hard gates or downgrade rules.

Plain-language note: a candidate can have strong popularity but still be downgraded if it has major fit, legal, reputation, or execution risks.

In [ ]:
def base_grade(score):
    if score >= 85:
        return "S"
    if score >= 75:
        return "A"
    if score >= 65:
        return "B"
    if score >= 50:
        return "C"
    return "D"

grade_order = ["D", "C", "B", "A", "S"]

def downgrade(grade, n=1):
    return grade_order[max(0, grade_order.index(grade) - n)]

def hard_gate(row):
    reasons = []
    if row["Audience Overlap"] < 6:
        reasons.append("Audience overlap too low")
    if row["Content Fit"] < 8:
        reasons.append("Content fit too low")
    if row["Reputation Safety"] < 5:
        reasons.append("Reputation or legal safety too low")
    if row["fit_risk_rate"] > 0.5 and row["mention_content_count"] >= 5:
        reasons.append("Community fit-risk signal too high")
    if str(row["legal_risk"]).lower() in ["high"]:
        reasons.append("Legal risk high")
    return len(reasons) > 0, "; ".join(reasons)

def apply_grade_rules(row, score_col):
    base = base_grade(row[score_col])
    blocked, block_reason = hard_gate(row)
    if blocked:
        return "D", True, block_reason, ""

    levels = 0
    reasons = []
    if str(row["budget_risk"]).lower() in ["high"]:
        levels += 1
        reasons.append("High licensing or budget risk")
    if bool(row["competitor_used"]):
        levels += 1
        reasons.append("Recently used by competitor or category saturation")
    if row["negative_rate"] > 0.4 and row["mention_content_count"] >= 5:
        levels += 1
        reasons.append("High negative signal rate")
    return downgrade(base, levels), False, "", "; ".join(reasons)

def action_for_grade(grade, blocked):
    if blocked:
        return "Pause: review audience overlap, content fit, legal/reputation risk"
    if grade == "S":
        return "Proceed: business outreach, concept pitch, core-user test"
    if grade == "A":
        return "Prioritize: collect cost estimate, ROI model, creator feedback"
    if grade == "B":
        return "Keep as backup: suitable for lightweight collaboration or future slot"
    if grade == "C":
        return "Low priority: proceed only if cost is low"
    return "Not recommended"

def tag_row(row):
    tags = []
    if row["core_overlap_rate"] >= 0.25:
        tags.append("High core-user overlap")
    if row["weighted_core_overlap_rate"] >= 0.30:
        tags.append("High weighted core overlap")
    if row["channel_overlap_rate"] >= 0.20:
        tags.append("Channel activity overlap")
    if row["high_engagement_overlap_rate"] >= 0.20:
        tags.append("High-engagement overlap")
    if row["purchase_intent_rate"] > 0:
        tags.append("Purchase intent")
    if row["collab_expect_rate"] > 0:
        tags.append("Collaboration expectation")
    if row["ugc_rate"] > 0:
        tags.append("UGC potential")
    if row["negative_rate"] >= 0.4:
        tags.append("Negative signal risk")
    if row["fit_risk_rate"] > 0:
        tags.append("Fit-risk signal")
    if bool(row["competitor_used"]):
        tags.append("Competitor/category saturation")
    return " / ".join(tags)

def combine_auto_and_human(auto_scored_df, human_agg_df):
    df = auto_scored_df.merge(human_agg_df, on="target_interest", how="left")
    df["human_score_weighted"] = df["human_score_weighted"].fillna(df["final_score_auto"])
    df["human_evaluator_count"] = df["human_evaluator_count"].fillna(0).astype(int)
    a = AUTO_WEIGHT_IN_HUMAN_WEIGHTED_SCORE
    df["final_score_human_weighted"] = (df["final_score_auto"] * a + df["human_score_weighted"] * (1 - a)).round(2)

    rows = []
    for _, row in df.iterrows():
        row = row.copy()
        auto_grade, auto_blocked, auto_block_reason, auto_downgrade_reason = apply_grade_rules(row, "final_score_auto")
        human_grade, human_blocked, human_block_reason, human_downgrade_reason = apply_grade_rules(row, "final_score_human_weighted")
        row["base_grade_auto"] = base_grade(row["final_score_auto"])
        row["final_grade_auto"] = auto_grade
        row["base_grade_human_weighted"] = base_grade(row["final_score_human_weighted"])
        row["final_grade_human_weighted"] = human_grade
        row["is_hard_blocked_auto"] = auto_blocked
        row["block_reason_auto"] = auto_block_reason
        row["downgrade_reason_auto"] = auto_downgrade_reason
        row["is_hard_blocked_human_weighted"] = human_blocked
        row["block_reason_human_weighted"] = human_block_reason
        row["downgrade_reason_human_weighted"] = human_downgrade_reason
        row["tags"] = tag_row(row)
        row["recommended_action_auto"] = action_for_grade(auto_grade, auto_blocked)
        row["recommended_action_human_weighted"] = action_for_grade(human_grade, human_blocked)
        rows.append(row.to_dict())
    return pd.DataFrame(rows).sort_values(by="final_score_human_weighted", ascending=False).reset_index(drop=True)

final_df = combine_auto_and_human(auto_scored_df, human_agg_df)

summary_cols = [
    "target_interest",
    "final_score_auto", "base_grade_auto", "final_grade_auto",
    "final_score_human_weighted", "human_score_weighted", "base_grade_human_weighted", "final_grade_human_weighted",
    "Audience Overlap", "Content Fit", "Commercial Potential", "Reputation Safety", "Community Spread", "Execution Feasibility",
    "core_overlap_rate", "weighted_core_overlap_rate", "channel_overlap_rate", "high_engagement_overlap_rate", "lift_core_vs_base",
    "purchase_intent_rate", "collab_expect_rate", "negative_rate", "fit_risk_rate", "ugc_rate",
    "tags", "recommended_action_human_weighted"
]
final_df[summary_cols]


## 18. Generate report text

This block creates readable summaries for each candidate IP.

Plain-language note: this turns tables into stakeholder-friendly explanation.

In [ ]:
def explain_candidate(row):
    return f"""
IP: {row['target_interest']}

Automated score: {row['final_score_auto']} | base grade: {row['base_grade_auto']} | final grade: {row['final_grade_auto']}
Human-weighted score: {row['final_score_human_weighted']} | weighted human score: {row['human_score_weighted']} | final grade: {row['final_grade_human_weighted']}

Score dimensions: Audience Overlap {row['Audience Overlap']}/25; Content Fit {row['Content Fit']}/20; Commercial Potential {row['Commercial Potential']}/20; Reputation Safety {row['Reputation Safety']}/15; Community Spread {row['Community Spread']}/10; Execution Feasibility {row['Execution Feasibility']}/10.
Audience overlap: base-game core users {int(row['base_core_count'])}, IP core users {int(row['ip_core_count'])}, overlapping core users {int(row['core_overlap_count'])}; core overlap rate {row['core_overlap_rate']:.1%}; weighted core overlap rate {row['weighted_core_overlap_rate']:.1%}.
Channel and high-engagement overlap: channel overlap rate {row['channel_overlap_rate']:.1%}; high-engagement overlap rate {row['high_engagement_overlap_rate']:.1%}; core lift {row['lift_core_vs_base']:.2f}x.
Text signals: mention count {int(row['mention_content_count'])}; purchase-intent rate {row['purchase_intent_rate']:.1%}; collaboration-expectation rate {row['collab_expect_rate']:.1%}; negative rate {row['negative_rate']:.1%}; fit-risk rate {row['fit_risk_rate']:.1%}; UGC signal rate {row['ugc_rate']:.1%}.
Tags: {row['tags']}
Recommended action: {row['recommended_action_human_weighted']}
""".strip()

for _, row in final_df.iterrows():
    print(explain_candidate(row))
    print("-" * 130)


## 19. Export Excel report

This exports the final results and supporting sheets to Excel.

Plain-language note: the notebook is for analysis, while the Excel file is for sharing and review.

In [ ]:
output_path = "game_ip_collaboration_fit_score_public.xlsx"

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    final_df.to_excel(writer, index=False, sheet_name="final_scores")
    auto_scored_df.to_excel(writer, index=False, sheet_name="auto_scores")
    core_overlap_df.to_excel(writer, index=False, sheet_name="audience_overlap_features")
    user_tier_df.to_excel(writer, index=False, sheet_name="user_tiers")
    user_strength_df.to_excel(writer, index=False, sheet_name="user_strength")
    user_interest_df.to_excel(writer, index=False, sheet_name="user_interest_raw")
    content_df.to_excel(writer, index=False, sheet_name="content_raw")
    text_feature_df.to_excel(writer, index=False, sheet_name="text_features")
    core_overlap_evidence_df.to_excel(writer, index=False, sheet_name="overlap_evidence")
    manual_prior.to_excel(writer, index=False, sheet_name="manual_prior")
    human_agg_df.to_excel(writer, index=False, sheet_name="human_agg_scores")
    human_individual_df.to_excel(writer, index=False, sheet_name="human_individual_scores")

try:
    from google.colab import files
    files.download(output_path)
except Exception:
    print(f"Excel file saved locally as: {output_path}")


## 20. Chart: automated score vs human-weighted score

Plain-language note: this chart helps compare what the data says versus what reviewers say after stakeholder judgment.

In [ ]:
plot_df = final_df[["target_interest", "final_score_auto", "final_score_human_weighted"]].set_index("target_interest")
plot_df.plot(kind="bar", figsize=(10, 6))
plt.title("IP Collaboration Score: Automated vs Human-weighted")
plt.ylabel("Score")
plt.ylim(0, 100)
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()


## 21. Chart: core overlap, channel overlap, and high-engagement overlap

Plain-language note: this is the most important chart for audience fit. It shows whether the base game’s core users also overlap with each candidate IP’s audience.

In [ ]:
overlap_plot_df = final_df[["target_interest", "core_overlap_rate", "channel_overlap_rate", "high_engagement_overlap_rate"]].set_index("target_interest")
overlap_plot_df.plot(kind="bar", figsize=(11, 6))
plt.title("Core / Channel / High-engagement User Overlap")
plt.ylabel("Rate")
plt.ylim(0, max(0.1, overlap_plot_df.max().max() * 1.3))
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()


# Usage guide

## What to edit for real usage

1. Edit `base_game_config` with your own base game keywords and channels.
2. Edit `candidate_ips` with candidate IP keywords and channels.
3. Replace the fictional sample data with CSV/API data.
4. Update `manual_prior` with business priors.
5. Optionally update `human_scores` with stakeholder review scores.

## Important metrics

- `core_overlap_rate`: percentage of base-game core users who are also core users of the candidate IP.
- `weighted_core_overlap_rate`: same idea, but weighted by user strength.
- `channel_overlap_rate`: base-game channel-active users who are also candidate-IP channel-active users.
- `high_engagement_overlap_rate`: overlap among high-engagement users.
- `lift_core_vs_base`: how much more likely base-game core users are to be candidate-IP core users compared with the overall sample.
- `purchase_intent_rate`: signals such as buy, pull, skin, bundle, banner.
- `fit_risk_rate`: signals such as forced, weird, out of place.

## Summary

This model does not simply measure popularity. It measures whether the base game’s strongest community members naturally overlap with a candidate IP’s audience, then combines that with commercial, reputation, spread, and execution signals.